# Nighttime Pothole Detection: is day-to-night synthetic data a valid substitute for real night data?

**Claim under test.** Real night-time pothole images are scarce, so day images are *transformed* into synthetic night images and used for training. This notebook tests whether that works, with a controlled experiment in which **every model is the same 1-class YOLO11m trained to detect potholes** (no zero-shot COCO baseline).

**The three models (equal image count, same hyperparameters, same seeds)**

| Model | Trained on | Role |
|---|---|---|
| **Day** | the *original day* images | control: what you get without the transformation |
| **Night-synthetic** | the *same images after the day-to-night transformation* (same boxes) | the method under test |
| **Night-real** | real night images | ceiling: what real night data would give |

All three are tested on three held-out test sets: **day**, **night-synthetic** (same scenes as the day test) and **real night**. This gives a 3x3 transfer matrix.

**What would prove the claim**
1. *Night-synthetic* works on real night (it is far above the Day model).
2. It recovers most of the *Night-real* ceiling.
3. The gain is caused by the transformation, because Day and Night-synthetic use identical scenes, boxes and image counts, so the transformation is the only difference.

**Pipeline**

| Step | What happens |
|---|---|
| 0 | Setup, config, helpers |
| 1 | EDA of the three sources: synthetic night, real night, original day |
| 2 | Real set: class check, pothole-only filter, remap to class 0 |
| 3 | Pair every synthetic night image with its original day image (stem, then content matching) and audit train/val/test leakage |
| 4 | Real set: perceptual-hash grouping, then a leak-free 70/15/15 split |
| 5 | Image processing: brightness/contrast alignment (`aligned`), before/after |
| 6 | Build the balanced benchmark (equal sizes, matched object-size mix), `raw` and `aligned` versions |
| 7 | Train the grid: {raw, aligned} x {day, night-synthetic, night-real} x seeds |
| 8 | Evaluation tooling (model-agnostic, with bootstrap CIs) |
| 9 | Results: 3x3 matrix, ceiling recovery, transformation gain with CI |
| 10 | Diagnostics, qualitative results, saved artifacts |

**Rules that keep the comparison fair**
* The real **test** set is never used for training, threshold selection or tuning.
* Each model's confidence threshold is chosen on **its own training domain's validation set** (the realistic setting: a model trained without real night data cannot use real night labels).
* The same preprocessing is applied to train and test images of all domains.
* Set `SMOKE_TEST = True` in the config cell for a fast dry run of the whole pipeline.

## 0. Setup

In [ ]:
!pip install -q -U ultralytics

In [ ]:
import os, sys, re, json, glob, shutil, random, time, yaml, warnings, gc, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image as IPyImage, FileLink
from tqdm.auto import tqdm
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ============================ CONFIG ============================
SMOKE_TEST = False                  # True = quick dry run of the whole pipeline

# ---- datasets --------------------------------------------------------------------------------
SYNTH_DIR = Path("/kaggle/input/datasets/phoenixz14/pothole-only")                         # synthetic NIGHT (transformed from day)
REAL_DIR  = Path("/kaggle/input/datasets/tavishisharma09/night-potholes-yolov8-format")    # REAL night
DAY_DIR   = Path("/kaggle/input/datasets/phoenixz14/daytime-images-3-classes/Original Dataset")                                 # ORIGINAL DAY images (IIT Madras pothole dataset). Add it via "Add Input" and set the path
REAL_POTHOLE_CLASS_ID = 1              # id of 'pothole' in the real set (class 0 is 'night-potholes', dropped)
DAY_POTHOLE_CLASS_IDS = None           # None = keep every box of the day set; else e.g. [0]

# ---- experiment grid ---------------------------------------------------------------------------
DOMAINS  = ["day", "night", "real"]    # day = original day, night = day->night synthetic, real = real night
PREPROCS = ["raw", "aligned"]          # image-processing conditions; applied identically to ALL domains, train and test
SEEDS    = [42, 43, 44]
SPLIT_FRACS = (0.70, 0.15, 0.15)       # real set train/val/test (grouped by near-duplicate)

# ---- image processing ('aligned') --------------------------------------------------------------
TARGET_MEAN = 90                       # gamma lifts dark images toward this mean grey level (never darkens)
GAMMA_MIN   = 0.4
CLAHE_CLIP  = 2.0

# ---- matching / leakage audit --------------------------------------------------------------------
HASH_THR    = 6                        # Hamming distance (of 64) at or below which two images count as near-duplicates
CORR_MIN    = 0.70                     # high-pass thumbnail correlation needed for content-based day<->night pairing
CORR_MARGIN = 0.10                     # best match must beat the runner-up by this much (guards against look-alike road scenes)
BOX_IOU_MIN = 0.70                     # boxes of a pair must agree (IoU) so labels can be shared
SIZE_BINS   = (0.005, 0.02, 0.08)      # image-level largest-box area cut points (fraction of image) used to match object scale

# ---- training ----------------------------------------------------------------------------------
MODEL_WEIGHTS = "yolo11m.pt"           # COCO-pretrained init for every model
IMGSZ, BATCH, EPOCHS, PATIENCE, CLOSE_MOSAIC = 640, 16, 60, 15, 8
RETRAIN = False                        # False = reuse an existing best.pt (lets you resume after a session timeout)
TIME_BUDGET_H = 9.5                    # stop launching new runs after this many hours; just re-run the cell to continue
EVAL_BATCH = 16
N_BOOT = 300                           # bootstrap resamples for confidence intervals
RUN_IMGSZ_SWEEP = True                 # oracle diagnostic (uses real VAL labels, clearly labelled as such)
N_CAP = None                           # None = use all; int = cap train images per domain (smoke test)
SEED = 42

if SMOKE_TEST:
    IMGSZ, BATCH, EPOCHS, PATIENCE, CLOSE_MOSAIC, SEEDS, N_CAP, N_BOOT = 320, 8, 2, 5, 1, [42], 40, 50

WORK = Path("/kaggle/working")
RUNS = WORK / "runs" / ("smoke" if SMOKE_TEST else "main")
BENCH = WORK / ("bench_smoke" if SMOKE_TEST else "bench")
FIGS = WORK / "figures"
for d in (RUNS, BENCH, FIGS):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# identical for every model: only the training images differ between runs
TRAIN_KW = dict(epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, optimizer="auto", cos_lr=True,
                close_mosaic=CLOSE_MOSAIC, warmup_epochs=3, mosaic=1.0, mixup=0.0, scale=0.7, fliplr=0.5, flipud=0.0,
                degrees=0.0, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, workers=2, device=DEVICE, plots=False,
                cache="ram", amp=True, deterministic=True, exist_ok=True)
print("device:", DEVICE, "| torch", torch.__version__, "| smoke:", SMOKE_TEST)

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_dataset_root(base):
    """Return the folder that directly contains train/valid/test (searches up to 2 levels down)."""
    base = Path(base)
    if not base.exists():
        raise FileNotFoundError(f"{base} not found. Contents of /kaggle/input: {os.listdir('/kaggle/input') if Path('/kaggle/input').exists() else 'n/a'}")
    lvl1 = [p for p in base.iterdir() if p.is_dir()]
    lvl2 = [q for p in lvl1 for q in p.iterdir() if q.is_dir()]
    for c in [base] + lvl1 + lvl2:
        if any((c / s / "images").is_dir() for s in ("train", "valid", "val", "test")):
            return c
    raise FileNotFoundError(f"No YOLO-style split folders (<split>/images) found under {base}")

def split_dirs(root):
    """{'train'|'val'|'test': (images_dir, labels_dir)} for the splits that exist."""
    out = {}
    for key, names in {"train": ["train"], "val": ["valid", "val"], "test": ["test"]}.items():
        for n in names:
            if (root / n / "images").is_dir():
                out[key] = (root / n / "images", root / n / "labels")
                break
    return out

def list_images(d):
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMAGE_EXTS)

def label_path_for(img_path):
    img_path = Path(img_path)
    return img_path.parent.parent / "labels" / f"{img_path.stem}.txt"

def collect_items(root):
    """{split: [(img_path, label_path)]}. YOLO layout if present, otherwise every image found under root (split 'all')."""
    root = Path(root)
    sd = {}
    try:
        sd = split_dirs(find_dataset_root(root))
    except FileNotFoundError:
        pass
    if sd:
        return {sp: [(p, ldir / f"{p.stem}.txt") for p in list_images(idir)] for sp, (idir, ldir) in sd.items()}
    lbls = {q.stem: q for q in root.rglob("*.txt")}
    imgs = sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    return {"all": [(p, lbls.get(p.stem)) for p in imgs]}

def read_yolo_labels(path, keep_ids=None):
    """Array (n,5): cls, xc, yc, w, h (normalised). Polygons become their enclosing box. keep_ids filters classes."""
    rows = []
    if path is not None and Path(path).exists():
        for line in Path(path).read_text().splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            v = [float(x) for x in p]
            if len(v) == 5:
                c, xc, yc, w, h = v
            else:
                xs, ys = v[1::2], v[2::2]
                x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                c, xc, yc, w, h = v[0], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
            rows.append([c, xc, yc, w, h])
    lab = np.array(rows, dtype=float).reshape(-1, 5)
    if keep_ids is not None:
        lab = lab[np.isin(lab[:, 0].astype(int), keep_ids)]
    return lab

def xywhn_to_xyxyn(lab):
    xc, yc, w, h = lab[:, 1], lab[:, 2], lab[:, 3], lab[:, 4]
    return np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], 1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def boxes_agree(a, b, thr=None):
    """True if two label arrays describe the same set of boxes (equal count, every box matched with IoU >= thr)."""
    thr = BOX_IOU_MIN if thr is None else thr
    if len(a) != len(b):
        return False
    if len(a) == 0:
        return True
    m = iou_matrix(xywhn_to_xyxyn(a), xywhn_to_xyxyn(b))
    return bool((m.max(1) >= thr).all() and (m.max(0) >= thr).all())

def mean_gray(path):
    return float(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE).mean())

def draw_boxes(img_bgr, lab, color=(0, 255, 0), thick=3):
    h, w = img_bgr.shape[:2]
    for _, xc, yc, bw, bh in lab:
        cv2.rectangle(img_bgr, (int((xc - bw / 2) * w), int((yc - bh / 2) * h)), (int((xc + bw / 2) * w), int((yc + bh / 2) * h)), color, thick)
    return img_bgr

def savefig(name):
    plt.savefig(FIGS / name, dpi=150, bbox_inches="tight")

def dhash_pair(path):
    """64-bit difference hash of an image and of its horizontal mirror (catches flipped augmentation copies)."""
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)
    h = (g[:, 1:] > g[:, :-1]).ravel()
    gf = g[:, ::-1]
    hf = (gf[:, 1:] > gf[:, :-1]).ravel()
    return h, hf

def hash_arrays(paths):
    hs, hfs = zip(*[dhash_pair(p) for p in paths]) if len(paths) else ([], [])
    return np.array(hs, dtype=bool).reshape(len(paths), 64), np.array(hfs, dtype=bool).reshape(len(paths), 64)

def min_hamming(Ha, Hb, Hb_flip, chunk=128):
    """For every row of Ha: smallest Hamming distance to any row of Hb (or its mirror). Returns (dist, index)."""
    best_d = np.full(len(Ha), 64); best_i = np.full(len(Ha), -1)
    if len(Hb) == 0:
        return best_d, best_i
    for s in range(0, len(Ha), chunk):
        a = Ha[s:s + chunk, None, :]
        d = np.minimum((a != Hb[None]).sum(2), (a != Hb_flip[None]).sum(2))
        best_i[s:s + chunk] = d.argmin(1); best_d[s:s + chunk] = d.min(1)
    return best_d, best_i

## 1. EDA of the three sources

Synthetic night (the transformed set), real night, and the original day set. The EDA reports counts, label health, object sizes and brightness, so the later benchmark can be balanced on evidence.

In [ ]:
def dataset_eda(root, name, keep_ids=None, n_bright=150):
    items = collect_items(root)
    print(f"=== EDA: {name} | {Path(root)} ===")
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
    for sp, its in items.items():
        classes, per_img, areas, sizes, empty, corrupt = Counter(), [], [], Counter(), 0, 0
        for p, l in its:
            try:
                with PILImage.open(p) as im:
                    sizes[im.size] += 1
            except Exception:
                corrupt += 1
                continue
            lab = read_yolo_labels(l)
            classes.update(lab[:, 0].astype(int).tolist())
            if keep_ids is not None:
                lab = lab[np.isin(lab[:, 0].astype(int), keep_ids)]
            per_img.append(len(lab)); empty += int(len(lab) == 0)
            areas += list(lab[:, 3] * lab[:, 4] * 100)
        n_lbl = sum(1 for _, l in its if l is not None and Path(l).exists())
        areas = np.array(areas) if areas else np.array([np.nan])
        print(f"\n--- SPLIT: {sp.upper()} ---")
        print(f"Images / label files : {len(its)} / {n_lbl}")
        print(f"Empty or no pothole  : {empty} | Corrupt images: {corrupt}")
        print(f"Classes (all boxes)  : {dict(sorted(classes.items()))}")
        print(f"Top image sizes      : {sizes.most_common(3)}")
        print(f"Boxes per image      : mean={np.mean(per_img):.2f} median={np.median(per_img):.0f} max={np.max(per_img)}")
        print(f"Box area (% of img)  : min={np.nanmin(areas):.3f} median={np.nanmedian(areas):.2f} max={np.nanmax(areas):.1f}")
        paths = [p for p, _ in its]
        sample = random.Random(SEED).sample(paths, min(n_bright, len(paths)))
        b = [mean_gray(p) for p in sample]
        print(f"Mean brightness      : {np.mean(b):.1f}")
        ax[0].hist(per_img, bins=np.arange(0, 23) - 0.5, alpha=0.5, density=True, label=sp)
        ax[1].hist(np.log10(areas[~np.isnan(areas)] + 1e-6), bins=40, alpha=0.5, density=True, label=sp)
        ax[2].hist(b, bins=30, alpha=0.5, density=True, label=sp)
    ax[0].set_title(f"{name}: potholes per image"); ax[1].set_title("log10 box area (% of image)"); ax[2].set_title("mean grey level (0 = black)")
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

dataset_eda(SYNTH_DIR, "synthetic night (transformed)")

In [ ]:
dataset_eda(REAL_DIR, "real night")

In [ ]:
if not DAY_DIR.exists():
    raise FileNotFoundError(
        f"DAY_DIR not found: {DAY_DIR}\n"
        "Add the original DAY dataset (IIT Madras pothole detection) to this notebook via 'Add Input', then set DAY_DIR in the config cell.\n"
        f"Datasets currently attached: {sorted(str(p) for p in Path('/kaggle/input').glob('*'))[:20]} / "
        f"{sorted(str(p) for p in Path('/kaggle/input/datasets').glob('*/*'))[:20] if Path('/kaggle/input/datasets').exists() else ''}")
dataset_eda(DAY_DIR, "original day", keep_ids=DAY_POTHOLE_CLASS_IDS)

## 2. Real night set: classes, pothole-only filtering

The real set has two class ids (`{1: 3055, 0: 330}`). The cell prints the names from `data.yaml` and shows examples of each class, so the pothole id can be confirmed visually. **Look at the class-0 examples**: if they are potholes too, set the config accordingly. All kept boxes are remapped to class 0, the same convention as the synthetic set.

In [ ]:
real_root = find_dataset_root(REAL_DIR)
real_sd = split_dirs(real_root)
print("real root:", real_root, "| splits found:", list(real_sd))

real_names = {}
for yp in list(real_root.glob("*.y*ml")) + list(Path(REAL_DIR).glob("*.y*ml")):
    d = yaml.safe_load(open(yp))
    if isinstance(d, dict) and "names" in d:
        n = d["names"]
        real_names = dict(enumerate(n)) if isinstance(n, list) else {int(k): v for k, v in n.items()}
        print(f"{yp.name}: names = {real_names}")
        break

cls_counts, per_class_examples = Counter(), {}
for sp, (idir, ldir) in real_sd.items():
    for p in list_images(idir):
        lab = read_yolo_labels(label_path_for(p))
        for row in lab:
            c = int(row[0]); cls_counts[c] += 1
            per_class_examples.setdefault(c, [])
            if len(per_class_examples[c]) < 40:
                per_class_examples[c].append((p, row))
print("boxes per class id:", dict(sorted(cls_counts.items())))

ids = sorted(per_class_examples)
fig, axes = plt.subplots(len(ids), 4, figsize=(16, 4 * len(ids)), squeeze=False)
rng = random.Random(SEED)
for r_i, c in enumerate(ids):
    picks = rng.sample(per_class_examples[c], min(4, len(per_class_examples[c])))
    for a_i in range(4):
        ax = axes[r_i][a_i]; ax.axis("off")
        if a_i >= len(picks):
            continue
        p, row = picks[a_i]
        img = draw_boxes(cv2.imread(str(p)), row[None, :])
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(f"class id {c} ({real_names.get(c, '?')})")
plt.tight_layout(); savefig("real_class_examples.png"); plt.show()

REAL_POTHOLE_IDS = [int(REAL_POTHOLE_CLASS_ID)]
print("Using pothole class id(s):", REAL_POTHOLE_IDS)

# keep every image with >=1 pothole box; boxes remapped to class 0
real_items = []     # dicts: path, lab (n,5 with class 0)
skipped = dropped_boxes = 0
for sp, (idir, ldir) in real_sd.items():
    for p in list_images(idir):
        lab = read_yolo_labels(label_path_for(p))
        keep = np.isin(lab[:, 0].astype(int), REAL_POTHOLE_IDS)
        dropped_boxes += int((~keep).sum())
        if keep.sum() == 0:
            skipped += 1; continue
        lab = lab[keep].copy(); lab[:, 0] = 0
        real_items.append({"path": p, "lab": lab})
print(f"real images kept: {len(real_items)} | images without potholes skipped: {skipped} | non-pothole boxes dropped: {dropped_boxes} | pothole boxes: {sum(len(r['lab']) for r in real_items)}")

## 3. Pair every synthetic night image with its original day image

The experiment's key control is that **Day and Night-synthetic see the same scenes with the same boxes**, so the transformation is the only difference. Pairing works in two stages:

1. **Filename match** after stripping Roboflow suffixes (`_jpg.rf.<hash>`) and common prefixes.
2. **Content match** for the rest: thumbnails are compared by zero-mean normalised correlation (robust to the brightness change), and a pair is accepted only if it is a mutual best match **and** the boxes agree (IoU >= `BOX_IOU_MIN`; skipped if the day set has no readable YOLO labels). Content matches must also beat the runner-up by `CORR_MARGIN`, so look-alike road scenes are not confused.

Pairs whose boxes do not agree are discarded, so both domains can share the same labels. A leakage audit then removes val/test images that have a near-duplicate (or mirrored copy) in train.

In [ ]:
def norm_stem(s):
    s = s.lower()
    s = re.sub(r"[_.]rf[._][0-9a-f]{8,}$", "", s)          # roboflow export hash
    s = re.sub(r"_(jpg|jpeg|png|bmp)$", "", s)
    s = re.sub(r"^(night|synthetic|synth|syn|dark|n2d|d2n)[-_]?", "", s)
    return s

def thumb(path, size=64):
    """High-pass, unit-norm grayscale thumbnail: correlation of two thumbnails ignores brightness/contrast changes and the shared sky/road gradient."""
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)
    g = (g - cv2.GaussianBlur(g, (0, 0), 4)).ravel()
    return g / (np.linalg.norm(g) + 1e-6)

night_items = {sp: [(p, read_yolo_labels(l)) for p, l in its] for sp, its in collect_items(SYNTH_DIR).items()}
day_pool = [(p, read_yolo_labels(l, DAY_POTHOLE_CLASS_IDS)) for its in collect_items(DAY_DIR).values() for p, l in its]
print({sp: len(v) for sp, v in night_items.items()}, "| day pool:", len(day_pool))

day_has_labels = any(len(l) for _, l in day_pool)
if not day_has_labels:
    print("[NOTE] no readable YOLO labels found for the day images (other annotation format?). Pairing will rely on filename/content only; "
          "labels are taken from the night set for BOTH domains anyway, so the experiment is unaffected.")
def agree(night_lab, day_lab):
    return True if not day_has_labels else boxes_agree(night_lab, day_lab)

day_by_stem = defaultdict(list)
for i, (p, _) in enumerate(day_pool):
    day_by_stem[norm_stem(p.stem)].append(i)

pairs, used_day, unmatched = [], set(), []
# ---- stage 1: filename ----
for sp, its in night_items.items():
    for p, lab in its:
        ok = [i for i in day_by_stem.get(norm_stem(p.stem), []) if i not in used_day and agree(lab, day_pool[i][1])]
        if ok:
            used_day.add(ok[0]); pairs.append({"split": sp, "night": p, "day": day_pool[ok[0]][0], "lab": lab, "stage": "filename"})
        else:
            unmatched.append((sp, p, lab))
print(f"stage 1 (filename): {len(pairs)} pairs | unmatched: {len(unmatched)}")

# ---- stage 2: content ----
free_day = [i for i in range(len(day_pool)) if i not in used_day]
if unmatched and free_day:
    S = np.stack([thumb(p) for _, p, _ in tqdm(unmatched, desc="thumbs night")])
    D = np.stack([thumb(day_pool[i][0]) for i in tqdm(free_day, desc="thumbs day")])
    C = S @ D.T
    best_d, best_s = C.argmax(1), C.argmax(0)
    second = np.partition(C, -2, axis=1)[:, -2] if C.shape[1] > 1 else np.full(len(C), -1.0)
    n2 = 0
    for k, (sp, p, lab) in enumerate(unmatched):
        j = int(best_d[k])
        if C[k, j] >= CORR_MIN and C[k, j] - second[k] >= CORR_MARGIN and best_s[j] == k and free_day[j] not in used_day and agree(lab, day_pool[free_day[j]][1]):
            used_day.add(free_day[j]); n2 += 1
            pairs.append({"split": sp, "night": p, "day": day_pool[free_day[j]][0], "lab": lab, "stage": "content"})
    print(f"stage 2 (content) : +{n2} pairs")

n_night_total = sum(len(v) for v in night_items.values())
rate = len(pairs) / max(n_night_total, 1)
tab = pd.DataFrame(pairs).groupby(["split", "stage"]).size().unstack(fill_value=0) if pairs else pd.DataFrame()
tab["night images"] = pd.Series({sp: len(v) for sp, v in night_items.items()})
print(f"\npaired {len(pairs)} / {n_night_total} night images ({100 * rate:.1f}%)"); display(tab)
if rate < 0.5:
    print("[WARNING] fewer than half the images were paired. Example normalised stems  night:",
          [norm_stem(p.stem) for _, p, _ in unmatched[:5]], " day:", [norm_stem(p.stem) for p, _ in day_pool[:5]])
    print("If the day set is not the exact source of the night images, or the transform changes geometry (crop/flip), pairing cannot work. Adjust norm_stem/CORR_MIN or the day dataset.")
assert len(pairs) > 50, "Too few day<->night pairs to run the experiment. Check DAY_DIR and the pairing settings above."

In [ ]:
# visual sanity check: night vs original day, same boxes drawn on both
rng = random.Random(SEED)
show = rng.sample(pairs, 4)
fig, axes = plt.subplots(4, 2, figsize=(10, 20))
for i, pr in enumerate(show):
    for j, key in enumerate(("day", "night")):
        im = draw_boxes(cv2.imread(str(pr[key])), pr["lab"])
        axes[i][j].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[i][j].axis("off")
        axes[i][j].set_title(f"{key} ({pr['stage']} match, {pr['split']})")
plt.tight_layout(); savefig("pairs_day_vs_night.png"); plt.show()
db = np.array([mean_gray(p["day"]) for p in show]); nb = np.array([mean_gray(p["night"]) for p in show])
print(f"mean brightness of these pairs: day {db.mean():.1f} -> night {nb.mean():.1f}")

In [ ]:
# leakage audit: val/test images that are near-duplicates (or mirrored copies) of a train image are removed
by_split = {sp: [p for p in pairs if p["split"] == sp] for sp in ("train", "val", "test")}
Htr, Htr_f = hash_arrays([p["night"] for p in by_split["train"]])
leak_report = {}
for sp in ("val", "test"):
    H, _ = hash_arrays([p["night"] for p in by_split[sp]])
    d, _ = min_hamming(H, Htr, Htr_f)
    leaky = d <= HASH_THR
    leak_report[sp] = (int(leaky.sum()), len(leaky))
    by_split[sp] = [p for p, bad in zip(by_split[sp], leaky) if not bad]
print({sp: f"{a}/{b} {sp} images had a near-duplicate in train and were removed" for sp, (a, b) in leak_report.items()})
print("paired, leak-free sizes:", {sp: len(v) for sp, v in by_split.items()})
assert min(len(v) for v in by_split.values()) > 10, "A split is nearly empty after leak removal. Check the split names (train/valid/test) of the synthetic set."

## 4. Real night set: leak-free train/val/test split

The real set only has a `train` folder, and exports from labelling tools often contain several augmented copies of one photo. Images are **grouped by perceptual hash (including mirrored copies)** and whole groups are assigned to a split, so no near-duplicate can sit in both train and test.

In [ ]:
def split_real(items, fracs, hash_thr, seed):
    paths = [it["path"] for it in items]
    H, Hf = hash_arrays(paths)
    n = len(items)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(n):
        dm = np.minimum((H[i][None] != H).sum(1), (H[i][None] != Hf).sum(1))
        for j in np.where(dm <= hash_thr)[0]:
            ri, rj = find(i), find(int(j))
            if ri != rj:
                parent[ri] = rj
    groups = defaultdict(list)
    for i in range(n):
        groups[find(i)].append(i)
    glist = list(groups.values())
    random.Random(seed).shuffle(glist)
    target = [f * n for f in fracs]; cur = [0, 0, 0]; assign = [[], [], []]
    for g in sorted(glist, key=len, reverse=True):       # big groups first, then fill the largest deficit
        k = int(np.argmax([target[s] - cur[s] for s in range(3)]))
        assign[k] += g; cur[k] += len(g)
    return assign, glist

assign, glist = split_real(real_items, SPLIT_FRACS, HASH_THR, SEED)
real_split = {sp: [real_items[i] for i in idx] for sp, idx in zip(("train", "val", "test"), assign)}
sizes = sorted((len(g) for g in glist), reverse=True)
print(f"{len(real_items)} images -> {len(glist)} near-duplicate groups | largest groups: {sizes[:6]} | singletons: {sum(s == 1 for s in sizes)}")
print({sp: f"{len(v)} images, {sum(len(r['lab']) for r in v)} boxes" for sp, v in real_split.items()})
sets_ = {sp: {str(r["path"]) for r in v} for sp, v in real_split.items()}
assert not (sets_["train"] & sets_["val"]) and not (sets_["train"] & sets_["test"]) and not (sets_["val"] & sets_["test"])
# audit: no cross-split near-duplicates remain
Htr_r, Htr_rf = hash_arrays([r["path"] for r in real_split["train"]])
for sp in ("val", "test"):
    Hs, _ = hash_arrays([r["path"] for r in real_split[sp]])
    dmin, _ = min_hamming(Hs, Htr_r, Htr_rf)
    print(f"real {sp}: images with a near-duplicate in train = {int((dmin <= HASH_THR).sum())} (should be 0)")

## 5. Image processing: brightness and contrast alignment

The EDA showed the synthetic night images are much darker than the real night images (mean grey about 54 against about 81). The `aligned` condition applies one fixed, label-free pipeline to **every image of every domain, at train and test time**:

1. **Gamma lift**: gamma is chosen per image so its mean grey level moves toward `TARGET_MEAN`. It only brightens (gamma <= 1), so bright day images are left alone.
2. **CLAHE** on the L channel (LAB space) to restore local contrast in the dark regions.

`raw` keeps the images untouched. Both conditions are trained and evaluated, so the notebook also measures what the image processing is worth.

In [ ]:
_CLAHE = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8))

def align_gamma(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    m = float(np.clip(g.mean(), 1e-3, 0.999)); t = TARGET_MEAN / 255
    return float(np.clip(np.log(t) / np.log(m), GAMMA_MIN, 1.0))

def align_image(img_bgr):
    gamma = align_gamma(img_bgr)
    lut = (255 * (np.arange(256) / 255.0) ** gamma).astype(np.uint8)
    img = cv2.LUT(img_bgr, lut)
    l, a, b = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2LAB))
    return cv2.cvtColor(cv2.merge((_CLAHE.apply(l), a, b)), cv2.COLOR_LAB2BGR)

# demo on one paired scene (day, night) and one real night image
demo = [("day (original)", cv2.imread(str(pairs[0]["day"]))), ("night (synthetic)", cv2.imread(str(pairs[0]["night"]))),
        ("real night", cv2.imread(str(real_items[0]["path"])))]
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
for i, (nm, im) in enumerate(demo):
    out = align_image(im)
    axes[i][0].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[i][0].set_title(f"{nm}: raw (mean {cv2.cvtColor(im, cv2.COLOR_BGR2GRAY).mean():.0f})")
    axes[i][1].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); axes[i][1].set_title(f"aligned (gamma {align_gamma(im):.2f}, mean {cv2.cvtColor(out, cv2.COLOR_BGR2GRAY).mean():.0f})")
    axes[i][0].axis("off"); axes[i][1].axis("off")
plt.tight_layout(); savefig("alignment_demo.png"); plt.show()

## 6. Build the balanced benchmark

* **Equal sizes.** Day, Night-synthetic and Night-real all get the same number of training images (`N_TRAIN`, the smaller of the paired pool and the real train split), and equal val and test sizes.
* **Same scenes for Day and Night-synthetic.** Day images are resized to their night counterpart's size and use the **same labels**, so the only difference is the transformation.
* **Matched object scale.** The paired training images are sampled so their *largest-box size mix* matches the real training set (`SIZE_BINS`). Real potholes are about 14x larger than the synthetic ones, and this keeps scale from dominating the comparison. The realised mix is printed, and where the paired pool lacks large potholes the shortfall is filled randomly and reported.
* Every image is decoded and re-encoded (JPEG q95) in all domains, so no domain has a compression advantage.

In [ ]:
rng = random.Random(SEED)
nb_bins = len(SIZE_BINS) + 1

def size_bin(lab):
    return 0 if len(lab) == 0 else int(np.searchsorted(SIZE_BINS, (lab[:, 3] * lab[:, 4]).max()))

def stratified_pick(bins, target_props, n, rng):
    byb = {b: [i for i, x in enumerate(bins) if x == b] for b in range(nb_bins)}
    for v in byb.values():
        rng.shuffle(v)
    picked = []
    for b in range(nb_bins):
        picked += byb[b][:int(round(n * target_props.get(b, 0)))]
    if len(picked) > n:
        rng.shuffle(picked); picked = picked[:n]
    elif len(picked) < n:
        taken = set(picked); rest = [i for i in range(len(bins)) if i not in taken]; rng.shuffle(rest); picked += rest[:n - len(picked)]
    return sorted(picked)

cap_tv = None if N_CAP is None else max(10, N_CAP // 2)
N_TRAIN = min(len(by_split["train"]), len(real_split["train"]), N_CAP or 10**9)
N_VAL   = min(len(by_split["val"]),   len(real_split["val"]),   cap_tv or 10**9)
N_TEST  = min(len(by_split["test"]),  len(real_split["test"]),  cap_tv or 10**9)
print(f"balanced sizes -> train {N_TRAIN} | val {N_VAL} | test {N_TEST} images per domain")

# real: random subset if the real split is larger than N
sel_real = {"train": sorted(rng.sample(range(len(real_split["train"])), N_TRAIN)),
            "val":   sorted(rng.sample(range(len(real_split["val"])), N_VAL)),
            "test":  sorted(rng.sample(range(len(real_split["test"])), N_TEST))}
real_sel = {sp: [real_split[sp][i] for i in idx] for sp, idx in sel_real.items()}

# paired: train is stratified to the real training object-size mix; val/test are random
real_bins = [size_bin(r["lab"]) for r in real_sel["train"]]
target_props = {b: real_bins.count(b) / len(real_bins) for b in range(nb_bins)}
pool_bins = [size_bin(p["lab"]) for p in by_split["train"]]
idx_tr = stratified_pick(pool_bins, target_props, N_TRAIN, rng)
pair_sel = {"train": [by_split["train"][i] for i in idx_tr],
            "val":   [by_split["val"][i]  for i in sorted(rng.sample(range(len(by_split["val"])),  N_VAL))],
            "test":  [by_split["test"][i] for i in sorted(rng.sample(range(len(by_split["test"])), N_TEST))]}
got = Counter(size_bin(p["lab"]) for p in pair_sel["train"])
print("object-size mix of the TRAIN sets (share of images per largest-box bin):")
print(pd.DataFrame({"real": target_props, "paired (day & night)": {b: got.get(b, 0) / N_TRAIN for b in range(nb_bins)}}).round(3).rename_axis("bin").T.to_string())
print("bins = largest box <", SIZE_BINS[0], "|", SIZE_BINS[0], "-", SIZE_BINS[1], "|", SIZE_BINS[1], "-", SIZE_BINS[2], "| >=", SIZE_BINS[2], "(fraction of image area)")

In [ ]:
import hashlib

def write_set(pre, dom, split, entries):
    """entries: list of dict(name, src, lab, size). Writes preprocessed JPEGs + class-0 labels; skips if already built identically."""
    root = BENCH / pre / dom / split
    sig = hashlib.md5((f"{pre}|{TARGET_MEAN}|{GAMMA_MIN}|{CLAHE_CLIP}|" + "|".join(f"{e['name']}:{e['src']}:{e['size']}" for e in entries)).encode()).hexdigest()
    marker = root / ".done"
    idir, ldir = root / "images", root / "labels"
    if not (marker.exists() and marker.read_text() == sig):
        if root.exists():
            shutil.rmtree(root)
        idir.mkdir(parents=True); ldir.mkdir(parents=True)
        for e in tqdm(entries, desc=f"{pre}/{dom}/{split}", leave=False):
            img = cv2.imread(str(e["src"]))
            if e["size"] is not None and (img.shape[1], img.shape[0]) != tuple(e["size"]):
                img = cv2.resize(img, tuple(e["size"]), interpolation=cv2.INTER_AREA)
            if pre == "aligned":
                img = align_image(img)
            cv2.imwrite(str(idir / f"{e['name']}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(ldir / f"{e['name']}.txt", "w") as f:
                for _, xc, yc, w, h in e["lab"]:
                    f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        marker.write_text(sig)
    return list_images(idir)

BENCH_SETS = {}
for pre in PREPROCS:
    for split in ("train", "val", "test"):
        night_e, day_e = [], []
        for k, pr in enumerate(pair_sel[split]):
            with PILImage.open(pr["night"]) as im:
                size = im.size
            nm = f"pair_{split}_{k:05d}"
            night_e.append({"name": nm, "src": pr["night"], "lab": pr["lab"], "size": None})
            day_e.append({"name": nm, "src": pr["day"], "lab": pr["lab"], "size": size})     # same boxes, resized to the night image's size
        real_e = [{"name": f"real_{split}_{k:05d}", "src": r["path"], "lab": r["lab"], "size": None} for k, r in enumerate(real_sel[split])]
        for dom, ent in (("day", day_e), ("night", night_e), ("real", real_e)):
            BENCH_SETS[(pre, dom, split)] = {"images": write_set(pre, dom, split, ent)}

YAMLS = {}
for pre in PREPROCS:
    for dom in DOMAINS:
        yp = BENCH / f"{dom}_{pre}.yaml"
        with open(yp, "w") as f:
            yaml.dump({"train": str(BENCH / pre / dom / "train" / "images"), "val": str(BENCH / pre / dom / "val" / "images"),
                       "test": str(BENCH / pre / dom / "test" / "images"), "nc": 1, "names": {0: "pothole"}}, f, sort_keys=False)
        YAMLS[(pre, dom)] = yp
print("benchmark ready:", BENCH)

In [ ]:
# proof of balance + effect of the image processing: one row per (domain, split)
rows = []
for dom in DOMAINS:
    for split in ("train", "val", "test"):
        imgs_raw = BENCH_SETS[("raw", dom, split)]["images"]
        labs = [read_yolo_labels(label_path_for(p)) for p in imgs_raw]
        areas = np.concatenate([l[:, 3] * l[:, 4] * 100 for l in labs])
        row = {"domain": dom, "split": split, "images": len(imgs_raw), "boxes": len(areas), "boxes/img": round(len(areas) / len(imgs_raw), 2),
               "median box %": round(float(np.median(areas)), 2)}
        for pre in PREPROCS:
            sample = random.Random(SEED).sample(BENCH_SETS[(pre, dom, split)]["images"], min(150, len(imgs_raw)))
            row[f"brightness[{pre}]"] = round(float(np.mean([mean_gray(p) for p in sample])), 1)
        rows.append(row)
bal = pd.DataFrame(rows).set_index(["domain", "split"])
display(bal)

fig, ax = plt.subplots(1, len(PREPROCS) + 1, figsize=(6 * (len(PREPROCS) + 1), 4.5))
for k, pre in enumerate(PREPROCS):
    for dom in DOMAINS:
        ax[k].hist([mean_gray(p) for p in BENCH_SETS[(pre, dom, "train")]["images"]], bins=30, alpha=0.5, density=True, label=dom)
    ax[k].set_title(f"train brightness [{pre}]"); ax[k].set_xlabel("mean grey level"); ax[k].legend()
for dom in DOMAINS:
    a = np.concatenate([read_yolo_labels(label_path_for(p))[:, 3] * read_yolo_labels(label_path_for(p))[:, 4] * 100 for p in BENCH_SETS[("raw", dom, "train")]["images"]])
    ax[-1].hist(np.log10(a), bins=40, alpha=0.5, density=True, label=dom)
ax[-1].set_title("train box area, log10 (% of image)"); ax[-1].legend()
plt.tight_layout(); savefig("benchmark_balance.png"); plt.show()

## 7. Train the grid

For every `seed` x `preprocessing` x `domain`, a fresh COCO-pretrained YOLO11m is trained **with identical hyperparameters** (`TRAIN_KW` in the config cell). Only the training images differ.

* Augmentation is the same everywhere (mosaic, scale 0.7, horizontal flip, mild HSV jitter) and deliberately does not randomise brightness enough to erase the day/night difference, so the domain effect stays measurable.
* Runs are cached: if `best.pt` exists the run is skipped, so after a Kaggle timeout simply re-run this cell. Seeds are the outer loop, so one complete seed finishes before the next starts.

In [ ]:
def run_name(pre, dom, seed): return f"{dom}_{pre}_s{seed}"
def best_path(pre, dom, seed): return RUNS / run_name(pre, dom, seed) / "weights" / "best.pt"

plan = [(seed, pre, dom) for seed in SEEDS for pre in PREPROCS for dom in DOMAINS]
t_start = time.time()
for seed, pre, dom in plan:
    if best_path(pre, dom, seed).exists() and not RETRAIN:
        print(f"[skip] {run_name(pre, dom, seed)} already trained"); continue
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print(f"time budget of {TIME_BUDGET_H} h reached. Re-run this cell to continue with the remaining runs."); break
    t0 = time.time()
    model = YOLO(MODEL_WEIGHTS)
    model.train(data=str(YAMLS[(pre, dom)]), seed=seed, project=str(RUNS), name=run_name(pre, dom, seed), **TRAIN_KW)
    print(f"{run_name(pre, dom, seed)} done in {(time.time() - t0) / 60:.1f} min")
    del model; gc.collect(); torch.cuda.empty_cache()

done = [r for r in plan if best_path(r[1], r[2], r[0]).exists()]
print(f"\ntrained runs available: {len(done)} / {len(plan)}")

In [ ]:
# training curves (validation mAP@0.5 on each model's OWN-domain val set; seed = first seed)
fig, ax = plt.subplots(1, len(PREPROCS), figsize=(7 * len(PREPROCS), 4.5), squeeze=False)
for k, pre in enumerate(PREPROCS):
    for dom in DOMAINS:
        f = RUNS / run_name(pre, dom, SEEDS[0]) / "results.csv"
        if f.exists():
            df_ = pd.read_csv(f); df_.columns = [c.strip() for c in df_.columns]
            ax[0][k].plot(df_["epoch"], df_["metrics/mAP50(B)"], label=f"{dom}-trained")
    ax[0][k].set_title(f"own-domain val mAP@0.5 [{pre}], seed {SEEDS[0]}"); ax[0][k].set_xlabel("epoch"); ax[0][k].legend()
plt.tight_layout(); savefig("training_curves.png"); plt.show()

## 8. Evaluation tooling

A model-agnostic evaluator (greedy IoU matching by confidence, 101-point AP, same convention as Ultralytics) that also stores **per-image** matches so we can bootstrap confidence intervals over test images. It is cross-checked against Ultralytics `val()` below.

* **mAP@0.5 and mAP@0.5:0.95** are threshold-free and are the headline metrics.
* **Operating point:** each model uses the confidence that maximises F1 on **its own training domain's validation set** (never real labels, unless it was trained on real).

In [ ]:
IOU_THRS = np.linspace(0.5, 0.95, 10)
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def gt_to_xyxy(lab, shape):
    h, w = shape
    xc, yc, bw, bh = lab[:, 1] * w, lab[:, 2] * h, lab[:, 3] * w, lab[:, 4] * h
    return np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], 1)

def predict_dataset(model, img_paths, conf=0.001, iou=0.7, imgsz=None):
    out = []
    for i in range(0, len(img_paths), EVAL_BATCH):
        chunk = [str(p) for p in img_paths[i:i + EVAL_BATCH]]
        for r in model.predict(chunk, conf=conf, iou=iou, imgsz=imgsz or IMGSZ, max_det=300, device=DEVICE, verbose=False):
            b = r.boxes
            out.append({"xyxy": b.xyxy.cpu().numpy(), "conf": b.conf.cpu().numpy(), "shape": r.orig_shape})
    return out

def match_image(xyxy, conf, g, thr):
    """Greedy match by descending confidence. Returns sorted conf, tp flags, and for each GT the conf of its matching detection (-1 if missed)."""
    order = np.argsort(-conf)
    tp = np.zeros(len(order), bool); gt_conf = -np.ones(len(g))
    if len(order) and len(g):
        ious = iou_matrix(xyxy[order], g); taken = np.zeros(len(g), bool)
        for k in range(len(order)):
            cand = np.where(taken, -1.0, ious[k]); j = int(cand.argmax())
            if cand[j] >= thr:
                tp[k] = True; taken[j] = True; gt_conf[j] = conf[order[k]]
    return conf[order], tp, gt_conf

def average_precision(conf, tp, n_gt):
    if n_gt == 0 or len(conf) == 0:
        return 0.0
    o = np.argsort(-conf); tp = tp[o]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    rec, prec = ctp / n_gt, ctp / (ctp + cfp)
    mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[1.0], prec, [0.0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_trapz(np.interp(x, mrec, mpre), x))

def evaluate_predictions(preds, img_paths, name=""):
    gts = [gt_to_xyxy(read_yolo_labels(label_path_for(p)), pr["shape"]) for p, pr in zip(img_paths, preds)]
    n_gt = int(sum(len(g) for g in gts))
    gt_area = np.concatenate([((g[:, 2] - g[:, 0]) * (g[:, 3] - g[:, 1])) / (pr["shape"][0] * pr["shape"][1]) for g, pr in zip(gts, preds)])
    aps, keep = [], None
    for thr in IOU_THRS:
        confs, tps, gtc = [], [], []
        for pr, g in zip(preds, gts):
            c, t, gc = match_image(pr["xyxy"], pr["conf"], g, thr)
            confs.append(c); tps.append(t); gtc.append(gc)
        aps.append(average_precision(np.concatenate(confs), np.concatenate(tps), n_gt))
        if abs(thr - 0.5) < 1e-9:
            keep = (confs, tps, gtc)
    confs, tps, gtc = keep
    return {"name": name, "n_images": len(img_paths), "n_gt": n_gt, "ap50": aps[0], "ap5095": float(np.mean(aps)),
            "conf50": np.concatenate(confs), "tp50": np.concatenate(tps), "gt_conf50": np.concatenate(gtc), "gt_area": gt_area,
            "img_conf": confs, "img_tp": tps, "img_ngt": [len(g) for g in gts]}

def op_point(res, thr):
    sel = res["conf50"] >= thr
    tp = int(res["tp50"][sel].sum()); fp = int(sel.sum()) - tp; fn = res["n_gt"] - tp
    P = tp / max(tp + fp, 1); R = tp / max(res["n_gt"], 1)
    return {"conf": float(thr), "P": P, "R": R, "F1": 2 * P * R / (P + R + 1e-9), "TP": tp, "FP": fp, "FN": fn}

def best_f1_conf(res, grid=np.linspace(0.02, 0.95, 94)):
    f1 = [op_point(res, t)["F1"] for t in grid]
    return float(grid[int(np.argmax(f1))]) if max(f1) > 0 else 0.25

def recall_by_size(res, thr):
    out = {}
    for lbl, (lo, hi) in {"small (<1% of image)": (0, 0.01), "medium (1-10%)": (0.01, 0.10), "large (>=10%)": (0.10, 9)}.items():
        m = (res["gt_area"] >= lo) & (res["gt_area"] < hi)
        out[lbl] = (int(m.sum()), float((res["gt_conf50"][m] >= thr).mean()) if m.any() else float("nan"))
    return out

def ultra_val(model, yaml_path, split, name):
    m = model.val(data=str(yaml_path), split=split, imgsz=IMGSZ, batch=EVAL_BATCH, conf=0.001, iou=0.7, device=DEVICE,
                  plots=False, verbose=False, project=str(RUNS / "eval"), name=name, exist_ok=True)
    return {"P": float(m.box.mp), "R": float(m.box.mr), "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map)}

## 9. Results

Every trained model is evaluated on the **day, night-synthetic and real-night test sets** (all three have the same size). Thresholds come from each model's own-domain validation set.

In [ ]:
RES, THR = {}, {}
for seed, pre, dom in tqdm(done, desc="evaluating"):
    model = YOLO(str(best_path(pre, dom, seed)))
    for tdom in DOMAINS:
        paths = BENCH_SETS[(pre, tdom, "test")]["images"]
        RES[(pre, dom, seed, tdom)] = evaluate_predictions(predict_dataset(model, paths), paths, f"{dom}-trained | {tdom} test")
    vp = BENCH_SETS[(pre, dom, "val")]["images"]
    THR[(pre, dom, seed)] = best_f1_conf(evaluate_predictions(predict_dataset(model, vp), vp, "val"))
    del model; gc.collect(); torch.cuda.empty_cache()

rec = []
for (pre, dom, seed, tdom), r in RES.items():
    op = op_point(r, THR[(pre, dom, seed)])
    rec.append({"pre": pre, "train": dom, "seed": seed, "test": tdom, "mAP50": r["ap50"], "mAP50-95": r["ap5095"],
                "conf": op["conf"], "P": op["P"], "R": op["R"], "F1": op["F1"], "TP": op["TP"], "FP": op["FP"], "FN": op["FN"]})
df_res = pd.DataFrame(rec)

def mean_std(x): return f"{x.mean():.3f} ± {x.std(ddof=0):.3f}"
for metric in ("mAP50", "mAP50-95"):
    print(f"\n===== {metric}: rows = training data, columns = test set (mean ± std over {len(SEEDS)} seeds) =====")
    for pre in PREPROCS:
        sub = df_res[df_res.pre == pre]
        tab = sub.pivot_table(index="train", columns="test", values=metric, aggfunc=mean_std).reindex(index=DOMAINS, columns=DOMAINS)
        print(f"\n[{pre}]"); display(tab)

In [ ]:
# cross-check of the custom evaluator against Ultralytics val() (night-trained model, own-domain test, first seed)
pre0, seed0 = PREPROCS[0], SEEDS[0]
if best_path(pre0, "night", seed0).exists():
    m_ = YOLO(str(best_path(pre0, "night", seed0)))
    yp = BENCH / f"_check_night_{pre0}.yaml"
    d_ = BENCH / pre0 / "night" / "test" / "images"
    yaml.dump({"train": str(d_), "val": str(d_), "test": str(d_), "nc": 1, "names": {0: "pothole"}}, open(yp, "w"))
    u = ultra_val(m_, yp, "test", "check_night_test")
    c = RES[(pre0, "night", seed0, "night")]
    print(f"night-trained / night test [{pre0}, seed {seed0}] custom: mAP50 {c['ap50']:.4f}, mAP50-95 {c['ap5095']:.4f} | ultralytics: {u['mAP50']:.4f} / {u['mAP50-95']:.4f}")
    del m_; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# grouped bars: AP50 of each training domain on each test set
fig, ax = plt.subplots(1, len(PREPROCS), figsize=(7 * len(PREPROCS), 4.8), squeeze=False)
cols = {"day": "#c9a227", "night": "#3b5ba9", "real": "#3a8f5c"}
for k, pre in enumerate(PREPROCS):
    sub = df_res[df_res.pre == pre]
    for j, dom in enumerate(DOMAINS):
        g = sub[sub.train == dom].groupby("test")["mAP50"]
        m_, s_ = g.mean().reindex(DOMAINS), g.std(ddof=0).reindex(DOMAINS)
        b = ax[0][k].bar(np.arange(3) + (j - 1) * 0.27, m_.values, 0.27, yerr=s_.values, capsize=3, label=f"trained on {dom}", color=cols[dom])
        ax[0][k].bar_label(b, fmt="%.2f", fontsize=8)
    ax[0][k].set_xticks(range(3)); ax[0][k].set_xticklabels([f"{d} test" for d in DOMAINS]); ax[0][k].set_ylim(0, 1)
    ax[0][k].set_title(f"mAP@0.5 [{pre}]"); ax[0][k].legend(fontsize=8)
plt.tight_layout(); savefig("transfer_matrix.png"); plt.show()

In [ ]:
# THE CLAIM, quantified on the REAL night test set
def boot_ap50(res_list, idx):
    vals = []
    for r in res_list:
        conf = np.concatenate([r["img_conf"][i] for i in idx]); tp = np.concatenate([r["img_tp"][i] for i in idx])
        vals.append(average_precision(conf, tp, int(sum(r["img_ngt"][i] for i in idx))))
    return float(np.mean(vals))

brng = np.random.default_rng(SEED)
claim_rows = []
for pre in PREPROCS:
    L = {dom: [RES[(pre, dom, s, "real")] for s in SEEDS if (pre, dom, s, "real") in RES] for dom in DOMAINS}
    if not all(L.values()):
        continue
    n_img = L["night"][0]["n_images"]
    d_gain, r_night, r_day, ap_n, ap_d, ap_r = [], [], [], [], [], []
    for _ in range(N_BOOT):
        idx = brng.integers(0, n_img, n_img)
        a_d, a_n, a_r = (boot_ap50(L[d], idx) for d in ("day", "night", "real"))
        d_gain.append(a_n - a_d); r_night.append(a_n / max(a_r, 1e-9)); r_day.append(a_d / max(a_r, 1e-9))
    ap = {d: np.mean([r["ap50"] for r in L[d]]) for d in DOMAINS}
    lo, hi = np.percentile(d_gain, [2.5, 97.5])
    claim_rows.append({"preprocessing": pre,
        "Day -> real night (mAP50)": ap["day"], "Night-synthetic -> real night": ap["night"], "Night-real -> real night (ceiling)": ap["real"],
        "transformation gain (night - day)": ap["night"] - ap["day"], "gain 95% CI": f"[{lo:.3f}, {hi:.3f}]", "P(gain > 0)": float(np.mean(np.array(d_gain) > 0)),
        "ceiling recovered: night": ap["night"] / max(ap["real"], 1e-9), "ceiling recovered: day": ap["day"] / max(ap["real"], 1e-9)})
claims = pd.DataFrame(claim_rows).set_index("preprocessing")
display(claims.round(3).T)

# preprocessing effect on the night-synthetic model (real test)
if len(PREPROCS) > 1:
    g = df_res[(df_res.train == "night") & (df_res.test == "real")].groupby("pre")["mAP50"]
    print("\nNight-synthetic -> real night, mAP50 by preprocessing (mean ± std over seeds):")
    print(g.agg(["mean", lambda x: x.std(ddof=0)]).rename(columns={"<lambda_0>": "std"}).round(3).to_string())

In [ ]:
# precision-recall curves on the real night test set (first seed)
def pr_curve(res):
    o = np.argsort(-res["conf50"]); tp = res["tp50"][o]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    return ctp / max(res["n_gt"], 1), ctp / np.maximum(ctp + cfp, 1)

fig, ax = plt.subplots(1, len(PREPROCS), figsize=(6.5 * len(PREPROCS), 5), squeeze=False)
for k, pre in enumerate(PREPROCS):
    for dom in DOMAINS:
        key = (pre, dom, SEEDS[0], "real")
        if key in RES:
            r_, p_ = pr_curve(RES[key]); ax[0][k].plot(r_, p_, color=cols[dom], label=f"trained on {dom} (AP50={RES[key]['ap50']:.3f})")
    ax[0][k].set_xlabel("Recall"); ax[0][k].set_ylabel("Precision"); ax[0][k].set_xlim(0, 1); ax[0][k].set_ylim(0, 1.02)
    ax[0][k].set_title(f"PR @ IoU 0.5 on REAL night test [{pre}]"); ax[0][k].grid(alpha=.3); ax[0][k].legend(fontsize=8)
plt.tight_layout(); savefig("pr_curves_real.png"); plt.show()

## 10. Diagnostics, qualitative results, artifacts

* Recall by object size on the real test set, at each model's own-domain threshold.
* **Oracle image-size sweep** (clearly labelled: it uses real validation labels, so it is *not* part of the claim). It shows whether inference resolution can compensate for the object-scale gap.
* Qualitative grid on real night images.

In [ ]:
rows = []
for (pre, dom, seed, tdom), r in RES.items():
    if tdom != "real":
        continue
    for lbl, (n, rc) in recall_by_size(r, THR[(pre, dom, seed)]).items():
        rows.append({"pre": pre, "train": dom, "size": lbl, "GT boxes": n, "recall": rc})
sz = pd.DataFrame(rows).groupby(["pre", "train", "size"]).agg({"GT boxes": "first", "recall": "mean"}).round(3)
display(sz.unstack("train")["recall"].reindex(columns=DOMAINS))

In [ ]:
if RUN_IMGSZ_SWEEP:
    print("ORACLE diagnostic: uses real VAL labels to look at inference image size. Not part of the main claim.")
    sweep = []
    for pre in PREPROCS:
        if not best_path(pre, "night", SEEDS[0]).exists():
            continue
        m_ = YOLO(str(best_path(pre, "night", SEEDS[0])))
        vp = BENCH_SETS[(pre, "real", "val")]["images"]
        for s in sorted({int(round(f * IMGSZ / 32) * 32) for f in (0.5, 0.75, 1.0, 1.25)}):
            sweep.append({"pre": pre, "infer imgsz": s, "mAP50 (night-trained, real val)": evaluate_predictions(predict_dataset(m_, vp, imgsz=s), vp)["ap50"]})
        del m_; gc.collect(); torch.cuda.empty_cache()
    display(pd.DataFrame(sweep).round(3))

In [ ]:
VIS_PRE = PREPROCS[-1]
vis_models = {dom: YOLO(str(best_path(VIS_PRE, dom, SEEDS[0]))) for dom in DOMAINS if best_path(VIS_PRE, dom, SEEDS[0]).exists()}

def show_grid(paths, fname, title):
    cols_ = 1 + len(vis_models)
    fig, axes = plt.subplots(len(paths), cols_, figsize=(5 * cols_, 5 * len(paths)), squeeze=False)
    for i, p in enumerate(paths):
        ims = [("Ground truth", draw_boxes(cv2.imread(str(p)), read_yolo_labels(label_path_for(p))))]
        for dom, m in vis_models.items():
            c = THR[(VIS_PRE, dom, SEEDS[0])]
            ims.append((f"{dom}-trained (conf {c:.2f})", m.predict(str(p), conf=c, imgsz=IMGSZ, device=DEVICE, verbose=False)[0].plot(line_width=2)))
        for j, (t, im) in enumerate(ims):
            axes[i][j].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[i][j].axis("off"); axes[i][j].set_title(t)
    plt.suptitle(title, y=1.0); plt.tight_layout(); savefig(fname); plt.show()

real_test_imgs = BENCH_SETS[(VIS_PRE, "real", "test")]["images"]
show_grid(random.Random(24).sample(real_test_imgs, min(6, len(real_test_imgs))), "qualitative_real.png", f"Real night test images [{VIS_PRE}]")

In [ ]:
night_test_imgs = BENCH_SETS[(VIS_PRE, "night", "test")]["images"]
show_grid(random.Random(SEED).sample(night_test_imgs, min(3, len(night_test_imgs))), "qualitative_night_synth.png", f"Synthetic night test images [{VIS_PRE}]")
del vis_models; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# automatic summary of the claim
print("=" * 78)
for pre, r in claims.iterrows():
    print(f"[{pre}]")
    print(f"  Trained on ORIGINAL DAY images       -> real night mAP50 = {r['Day -> real night (mAP50)']:.3f}")
    print(f"  Trained on TRANSFORMED NIGHT images  -> real night mAP50 = {r['Night-synthetic -> real night']:.3f}")
    print(f"  Trained on REAL NIGHT images (ceiling)-> real night mAP50 = {r['Night-real -> real night (ceiling)']:.3f}")
    print(f"  Transformation gain = {r['transformation gain (night - day)']:+.3f}  95% CI {r['gain 95% CI']}  P(gain>0) = {r['P(gain > 0)']:.2f}")
    print(f"  Synthetic night recovers {100 * r['ceiling recovered: night']:.0f}% of the real-data ceiling (day-trained: {100 * r['ceiling recovered: day']:.0f}%)")
print("=" * 78)

In [ ]:
df_res.round(4).to_csv(WORK / "results_long.csv", index=False)
claims.round(4).to_csv(WORK / "results_claims.csv")
sz.to_csv(WORK / "results_recall_by_size.csv")
bal.to_csv(WORK / "benchmark_balance.csv")
pd.DataFrame(pairs)[["split", "stage"]].value_counts().rename("n").reset_index().to_csv(WORK / "pairing_summary.csv", index=False)
summary = {"config": {"model": MODEL_WEIGHTS, "imgsz": IMGSZ, "epochs": EPOCHS, "batch": BATCH, "seeds": SEEDS, "preprocs": PREPROCS,
                      "target_mean": TARGET_MEAN, "clahe_clip": CLAHE_CLIP, "hash_thr": HASH_THR, "split_fracs": SPLIT_FRACS, "smoke_test": SMOKE_TEST},
           "sizes": {"train": N_TRAIN, "val": N_VAL, "test": N_TEST},
           "pairing": {"night_images": n_night_total, "paired": len(pairs)},
           "leak_removed": {k: list(v) for k, v in leak_report.items()},
           "thresholds": {f"{k[1]}|{k[0]}|s{k[2]}": v for k, v in THR.items()},
           "claims": claims.round(4).reset_index().to_dict("records")}
(WORK / "results_summary.json").write_text(json.dumps(summary, indent=2, default=str))

ZIP_PATH = WORK / "report_outputs.zip"
ZIP_PATH.unlink(missing_ok=True)
include = [WORK / f for f in ("results_long.csv", "results_claims.csv", "results_recall_by_size.csv", "benchmark_balance.csv", "pairing_summary.csv", "results_summary.json")]
include += sorted(FIGS.glob("*")) + sorted(BENCH.glob("*.yaml"))
include += [best_path(pre, dom, SEEDS[0]) for pre in PREPROCS for dom in DOMAINS if best_path(pre, dom, SEEDS[0]).exists()]
include += [RUNS / run_name(pre, dom, s) / "results.csv" for pre in PREPROCS for dom in DOMAINS for s in SEEDS if (RUNS / run_name(pre, dom, s) / "results.csv").exists()]
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in include:
        if Path(p).exists():
            z.write(p, Path(p).relative_to(WORK))
print(f"zip: {ZIP_PATH} | {ZIP_PATH.stat().st_size / 1e6:.1f} MB | {len(include)} files")
display(FileLink("report_outputs.zip", result_html_prefix="Click here to download: "))